# QDC Trading Workflow for One Round

Open and run this notebook from its own folder with Python 3.10 or later and pandas. It takes you from market data to proposed orders, manual review, and confirmed fills. The next round starts from the settled ledger. Use the [backtest](qdc-etf-50-backtest.ipynb) to compare rules and the [data notebook](qdc-etf-data-refresh.ipynb) to refresh and analyze market data.

By default, the notebook rehearses October 19, 2026 with historical prices and preset review decisions and fills. Each rehearsal starts with €10,000 and uses a temporary ledger. During the competition, set `MODE` to `live` and enter the trade date and the latest close expected to be available. The live ledger, `qdc-trading-ledger.sqlite3`, stays in this folder. Pick one language folder for the competition and use its ledger throughout.

## 1. Load quotes and choose the signal date

Read the official 50-instrument universe and the daily price CSV. `REFRESH_FROM_YAHOO` enables the same refresh used by the data notebook. Orders use closing prices available before the trade date. Check the latest date with complete quotes and any gaps in the signal window.

**Optional AI support.** Turn the data report into a review checklist grouped by ticker and date. The code checks coverage and dates.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import display

from qdc_trading import approve_round, load_market, plan_round, read_round, settle_round
from update_qdc_prices import ISSUE_FIELDS, latest_completed_date, load_history, refresh_prices, save_refresh

DATA_DIR = Path.cwd()
MODE = "rehearsal"  # Set to "live" during the competition.
REFRESH_FROM_YAHOO = False
TRADE_DATE = "2026-10-19"
EXPECTED_SIGNAL_DATE = "2026-10-16"
RULE = {"core_weight": 0.70, "cash_buffer": 0.01, "signal_sessions": 5}

if MODE not in {"rehearsal", "live"}:
    raise ValueError("Choose rehearsal or live.")
rehearsal_dir = TemporaryDirectory(prefix="qdc-rehearsal-") if MODE == "rehearsal" else None
ledger_path = Path(rehearsal_dir.name if rehearsal_dir else DATA_DIR) / "qdc-trading-ledger.sqlite3"

if REFRESH_FROM_YAHOO:
    cutoff = latest_completed_date()
    updated, issues = refresh_prices(load_history(DATA_DIR), cutoff)
    save_refresh(DATA_DIR, updated, issues, cutoff)
    print(f"Requested Yahoo Finance quotes through {cutoff}")
else:
    print("Using the saved Yahoo Finance snapshot")

issue_path = DATA_DIR / "qdc-50-data-issues.csv"
if issue_path.exists():
    data_issues = pd.read_csv(issue_path, usecols=ISSUE_FIELDS)
    print(f"The last refresh report has {len(data_issues)} rows to review")
    if not data_issues.empty:
        display(data_issues)

assets, all_closes, close, dividend, split = load_market(DATA_DIR)
market = {
    "close": close, "dividend": dividend, "split": split,
    "equity_tickers": assets.index[assets["core_eligible"]],
}
trade_day = pd.Timestamp(TRADE_DATE)
expected_signal_day = pd.Timestamp(EXPECTED_SIGNAL_DATE)
print(f"{len(assets)} instruments | Latest complete close {close.index[-1].date()}")

## 2. Read cash and holdings and propose orders

Start with €10,000 and no holdings in the first round. Later rounds read the cash balance and positions from the previous confirmed fills. By default, the core is chosen by its five-session total return and the companion by its five-session volatility. Enter the team's backtested core weight, cash reserve, and signal lookback in `RULE`. The proposed order is the target share count minus the current holding.

**Optional AI support.** Give AI the signal cutoff, account state, and proposed orders. Ask it to explain the proposed changes and flag anything the reviewer should inspect.

In [ ]:
opening, saved_approval, saved_settlement = read_round(ledger_path, TRADE_DATE)
if saved_approval is None:
    window_size = max(5, RULE["signal_sessions"]) + 1
    signal_dates = close.index[close.index < trade_day][-window_size:]
    observed_dates = all_closes.index[all_closes.index < trade_day]
    if not len(signal_dates) or signal_dates[-1] != expected_signal_day:
        raise ValueError("The latest complete close differs from the expected signal date. Check the data.")
    if not observed_dates[observed_dates >= signal_dates[0]].isin(signal_dates).all():
        raise ValueError("Some instruments are missing quotes within the signal window.")
    holdings = pd.Series(opening["holdings"], dtype="int64").reindex(close.columns, fill_value=0)
    signal_day, core, partner, target, orders = plan_round(
        trade_day, holdings, float(opening["cash"]), **market, **RULE
    )
    suggested = orders["delta"].to_dict()
    references = close.loc[signal_day].to_dict()
    print(f"Signal date {signal_day.date()} | Core {core} | Companion {partner}")
    display(orders.assign(target_shares=target.reindex(orders.index)).round(4))
else:
    suggested = saved_approval["suggested"]
    print(f"Loaded the approved round for signal date {saved_approval['signal_date']}")
print(f"Opening cash €{float(opening['cash']):,.2f} | Opening holdings {opening['holdings']}")

## 3. Review and lock the submission list

In live mode, check the ticker, ISIN, side, and share count. Fill in `APPROVED_ORDERS`, `REVIEW_NOTE`, and `APPROVAL_FOR_DATE`, then set `CONFIRM_APPROVAL = True` and run the cell. Use positive whole numbers to buy and negative whole numbers to sell. You can also approve an empty list. The code checks the five-order limit, available shares for sells, and estimated cash at reference prices.

The ledger stores both the proposal and the approved list. Submit the approved orders manually through the official channel. Page 8 of the Belfius slides gives two execution-price descriptions. The reference close estimates the budget here, and the confirmed fill determines the ledger entry.

**Optional AI support.** Draft a summary of the changes made in review and the reasons you entered. The reviewer decides the final submission.

In [ ]:
CONFIRM_APPROVAL = False
APPROVAL_FOR_DATE = ""
APPROVED_ORDERS = {}  # Example: {"XDWS.DE": 130, "LYOR.DE": 26}
REVIEW_NOTE = ""

if MODE == "rehearsal":
    CONFIRM_APPROVAL = True
    APPROVAL_FOR_DATE = TRADE_DATE
    APPROVED_ORDERS = {"XDWS.DE": 130, "LYOR.DE": 26}
    REVIEW_NOTE = "Rehearsal review. Reduce the core order from 134 shares to 130."

approval = saved_approval
if approval is None and CONFIRM_APPROVAL:
    if APPROVAL_FOR_DATE != TRADE_DATE:
        raise ValueError("The approval date must match the current trade date.")
    approval = approve_round(
        ledger_path, TRADE_DATE, opening, signal_day.date().isoformat(), RULE,
        suggested, APPROVED_ORDERS, references, REVIEW_NOTE,
    )
if approval is not None:
    reviewed = pd.DataFrame([
        {"Ticker": ticker, "ISIN": assets.at[ticker, "isin"],
         "Proposed shares": suggested.get(ticker, 0), "Approved shares": approval["approved"].get(ticker, 0)}
        for ticker in dict.fromkeys([*suggested, *approval["approved"]])
    ], columns=["Ticker", "ISIN", "Proposed shares", "Approved shares"])
    print(f"Review saved. {approval['review_note']}")
    display(reviewed)
else:
    print("Awaiting manual approval.")

## 4. Record confirmed fills and update the account

Once the complete fill report arrives, enter the executed share count and average price in euros for each approved order. Use the actual share count for a partial fill and `(0, None)` for an unfilled order. Enter `FILL_REPORT_FOR_DATE` and set `CONFIRM_FILL_REPORT = True` once you have checked the report. Confirmed dividends, fees, and splits can be recorded as separate adjustments.

The code validates the direction and size of each fill, updates cash and holdings, and records the same report only once. The rehearsal fills 120 of the 130 submitted core shares and all 26 companion shares at their historical closes.

**Optional AI support.** Extract tickers, share counts, and prices into a draft. Compare the draft with the original report before confirming settlement.

In [ ]:
CONFIRM_FILL_REPORT = False
FILL_REPORT_FOR_DATE = ""
FILL_REPORT = {}  # Example: {"XDWS.DE": (120, "51.24"), "LYOR.DE": (26, "110.272")}
CASH_ADJUSTMENT = "0"
SHARE_ADJUSTMENTS = {}

if MODE == "rehearsal":
    CONFIRM_FILL_REPORT = True
    FILL_REPORT_FOR_DATE = TRADE_DATE
    FILL_REPORT = {"XDWS.DE": (120, "51.24"), "LYOR.DE": (26, "110.272")}
    for ticker, (_, price) in FILL_REPORT.items():
        assert abs(close.at[trade_day, ticker] - float(price)) < 0.0001

settlement = saved_settlement
if CONFIRM_FILL_REPORT:
    if FILL_REPORT_FOR_DATE != TRADE_DATE:
        raise ValueError("The fill report date must match the current trade date.")
    settlement = settle_round(
        ledger_path, TRADE_DATE, FILL_REPORT,
        cash_adjustment=CASH_ADJUSTMENT, share_adjustments=SHARE_ADJUSTMENTS,
    )
if settlement is None:
    print("Awaiting the complete fill report.")
else:
    display(pd.DataFrame([
        {"Ticker": ticker, "Proposed shares": suggested.get(ticker, 0),
         "Approved shares": approval["approved"][ticker], "Filled shares": quantity,
         "Average fill price": price}
        for ticker, (quantity, price) in settlement["fills"].items()
    ], columns=["Ticker", "Proposed shares", "Approved shares", "Filled shares", "Average fill price"]))
    account_after = settlement["account_after"]
    print(f"Closing cash €{float(account_after['cash']):,.2f} | Closing holdings {account_after['holdings']}")

## 5. Carry the settled account into the next round

The rehearsal reads the opening balance and holdings for October 22 from the same ledger and generates the next set of orders. In live mode, enter the next trade date and signal date, then rerun the preceding sections. Keep the same ledger and use the approval and fill switches as each round progresses.

In [ ]:
if MODE == "rehearsal":
    next_date = "2026-10-22"
    next_opening, _, _ = read_round(ledger_path, next_date)
    assert next_opening == settlement["account_after"]
    next_holdings = pd.Series(next_opening["holdings"], dtype="int64").reindex(close.columns, fill_value=0)
    next_signal, _, _, _, next_orders = plan_round(
        pd.Timestamp(next_date), next_holdings, float(next_opening["cash"]), **market, **RULE
    )
    print(f"Next round {next_date} | Signal date {next_signal.date()} | Cash carried forward €{float(next_opening['cash']):,.2f}")
    display(next_orders.round(4))